# Warpmate, Phase 2: CUDA basics, GPU solver, benchmarks

This notebook builds Warpmate on a free Colab GPU and runs:

1. **CUDA basics:** small practice programs (adding and squaring numbers on the GPU)
2. **GPU solver:** solves king + rook vs king on the GPU and checks it matches the CPU exactly
3. **Benchmarks:** single-threaded CPU vs multithreaded CPU vs GPU on this machine

**Before you start:** *Runtime → Change runtime type → T4 GPU → Save*, then run the cells top to bottom.

In [ ]:
# Which GPU did Colab give us?
!nvidia-smi

## Get the code

Either upload `warpmate-colab.zip` (drag it into the **Files** panel on the left, or run the upload cell below), or skip the upload and the next cell clones the code from GitHub.

In [ ]:
# Optional: upload warpmate-colab.zip from your computer.
from google.colab import files
files.upload()

In [ ]:
import os
os.chdir('/content')
if os.path.exists('/content/warpmate-colab.zip'):
    !rm -rf Warpmate && mkdir Warpmate && unzip -q warpmate-colab.zip -d Warpmate
    print('Using the uploaded zip')
else:
    !rm -rf Warpmate && git clone -q https://github.com/vasishtk1/Warpmate.git
    print('Cloned from GitHub')
%cd /content/Warpmate

## Build

The build turns on GPU support automatically when it finds the CUDA compiler (`nvcc`). Look for `Warpmate GPU support: ON`.

In [ ]:
import os
if os.path.exists('/usr/local/cuda/bin/nvcc'):
    os.environ['CUDACXX'] = '/usr/local/cuda/bin/nvcc'
!nvcc --version | tail -2
!cmake -S . -B build -DCMAKE_BUILD_TYPE=Release 2>&1 | tail -4
!cmake --build build -j 2>&1 | tail -8

## Step 3: CUDA basics

Prints your GPU's specs, then adds and squares millions of numbers on the GPU, timing each part (copying data in, running the kernel, copying results out) and checking the answers against the CPU.

In [ ]:
!./build/cuda-basics

## Step 4: GPU solver

Runs all tests, including `gpu`, which solves KRvK on both the CPU and the GPU and requires the two tables to be **byte-for-byte identical**. Then solves on the GPU with round-by-round output, and looks up the 16-move position in the GPU-built table.

In [ ]:
!ctest --test-dir build --output-on-failure

In [ ]:
!./build/warpmate-solve --gpu -o tables/krvk_gpu.wmtb
!./build/warpmate-probe -t tables/krvk_gpu.wmtb "8/8/8/8/8/2k5/1R6/K7 w - - 0 1" | grep -E "Result|Best|Perfect"

## Step 5: Benchmarks

Times each solver 5 times on this machine and reports the median and best. The GPU gets an untimed warm-up run first. KRvK is small (half a million positions), so expect the GPU's advantage to be modest here; bigger endings come later.

In [ ]:
!echo "CPU: $(lscpu | grep 'Model name' | sed 's/Model name: *//') ($(nproc) threads)"
!./build/warpmate-bench --trials 5

## Optional: profile with Nsight Systems

Shows where the GPU time goes (each kernel, memory copies). Skips itself if `nsys` isn't installed on this Colab machine.

In [ ]:
!if command -v nsys >/dev/null; then nsys profile --stats=true --force-overwrite true -o krk_profile ./build/warpmate-solve --gpu -o /tmp/krk.wmtb 2>&1 | tail -60; else echo "nsys not installed here, skipping"; fi

## Done

Copy the output of the **Build**, **Step 3**, **Step 4** and **Step 5** cells and paste it back.